In [1]:
from dotenv import find_dotenv, load_dotenv
from getpass import getpass
import os

load_dotenv(find_dotenv(), override=True)
#os.environ["OPENAI_MODEL_ID"] = "openai/gpt-4o-2024-08-06"
os.environ["OPENAI_MODEL_ID"] = "openai/gpt-3.5-turbo-0125"
os.environ.pop("GEMINI_API_KEY", None)

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OPENAI_API_KEY:")

In [5]:
import sqlite3
import pandas as pd
from crewai.tools import tool
from crewai import Agent, Task, Crew, Process
from crewai import LLM
import time
from functools import wraps
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from datetime import datetime

# ============================================
# Global Walk-Forward State
# ============================================
PIPELINE_STATE = {
    "current_eval_date": None,  
    "allow_returns": True       
}

# ============================================
# Rate Limiting & Tools Setup
# ============================================
class RateLimiter:
    def __init__(self, calls_per_minute=5):
        self.calls_per_minute = calls_per_minute
        self.interval = 60.0 / calls_per_minute
        self.last_call = 0
        self.call_count = 0
        
    def wait_if_needed(self):
        current_time = time.time()
        time_since_last = current_time - self.last_call
        if time_since_last < self.interval:
            sleep_time = self.interval - time_since_last
            time.sleep(sleep_time)
        self.last_call = time.time()
        self.call_count += 1

rate_limiter = RateLimiter(calls_per_minute=5)

def rate_limit(func):
    @wraps(func)
    @retry(
        retry=retry_if_exception_type((Exception,)),
        wait=wait_exponential(multiplier=1, min=4, max=60),
        stop=stop_after_attempt(5)
    )
    def wrapper(*args, **kwargs):
        rate_limiter.wait_if_needed()
        try:
            return func(*args, **kwargs)
        except Exception as e:
            raise
    return wrapper

@tool("query_firm_database")
@rate_limit
def query_firm_database(sql_query: str) -> str:
    """Execute SQL queries on the firms database."""
    query_lower = sql_query.lower()
    if not PIPELINE_STATE["allow_returns"] and "ret_fwd_1" in query_lower:
        return f"SECURITY BLOCK: Access to 'ret_fwd_1' is strictly prohibited during evaluation for {PIPELINE_STATE['current_eval_date']}."
    
    try:
        conn = sqlite3.connect('firm_data.db')
        df = pd.read_sql_query(sql_query, conn)
        conn.close()
        
        if not PIPELINE_STATE["allow_returns"] and "ret_fwd_1" in df.columns:
            df = df.drop(columns=["ret_fwd_1"])
            
        if df.empty: return "Query returned no results."
        result = f"Query returned {len(df)} rows:\n\n" + df.head(5).to_string(index=False)
        return result
    except Exception as e:
        return f"Error executing query: {str(e)}"

@tool("get_database_schema")
@rate_limit
def get_database_schema() -> str:
    """Get the schema of the firms database."""
    try:
        conn = sqlite3.connect('firm_data.db')
        cursor = conn.cursor()
        cursor.execute("PRAGMA table_info(firms)")
        columns = cursor.fetchall()
        conn.close()
        
        schema = "Database: firm_data.db\nTable: firms\nColumns:\n"
        for col in columns:
            if not PIPELINE_STATE["allow_returns"] and col[1] == "ret_fwd_1":
                continue 
            schema += f"  - {col[1]} ({col[2]})\n"
        return schema
    except Exception as e:
        return f"Error retrieving schema: {str(e)}"

@tool("test_complex_condition")
@rate_limit
def test_complex_condition(sql_condition: str, date: str = None) -> str:
    """
    Test complex conditions with AND/OR/NOT logic.
    CRITICAL: Pass ONLY the condition (e.g., 'bm > 1.0 AND mom12m < 0.5'). 
    DO NOT pass a full SELECT statement.
    For 'date', you can pass a single date ('YYYY-MM-DD') or a range ('YYYY-MM-DD to YYYY-MM-DD').
    """
    # Auto-correct if the LLM accidentally passes a full SQL query
    if "WHERE " in sql_condition.upper():
        sql_condition = sql_condition[sql_condition.upper().find("WHERE ") + 6:]
    elif "SELECT " in sql_condition.upper():
        return "Error: Please pass ONLY the condition string (e.g., 'mve > 1.0'). Do not pass a SELECT query."

    if not PIPELINE_STATE["allow_returns"] and "ret_fwd_1" in sql_condition.lower():
        return f"SECURITY BLOCK: You cannot use 'ret_fwd_1' during zero-shot evaluation for {PIPELINE_STATE['current_eval_date']}."
        
    try:
        conn = sqlite3.connect('firm_data.db')
        target_date = date if date else PIPELINE_STATE["current_eval_date"]
        
        # Parse date range vs single date
        if " to " in target_date:
            start_date, end_date = target_date.split(" to ")
            date_clause = f"datadate BETWEEN '{start_date.strip()}' AND '{end_date.strip()}'"
        else:
            date_clause = f"datadate = '{target_date.strip()}'"
        
        count_query = f"SELECT COUNT(*) as n_firms FROM firms WHERE {date_clause} AND ({sql_condition})"
        columns_to_select = "datadate, ticker, mve, bm, mom12m" if 'ticker' in sql_condition else "*"
        sample_query = f"SELECT {columns_to_select} FROM firms WHERE {date_clause} AND ({sql_condition}) LIMIT 5"
        
        n_firms = pd.read_sql_query(count_query, conn)['n_firms'].values[0]
        sample_df = pd.read_sql_query(sample_query, conn)
        conn.close()
        
        if not PIPELINE_STATE["allow_returns"] and "ret_fwd_1" in sample_df.columns:
            sample_df = sample_df.drop(columns=["ret_fwd_1"])
            
        result = f"Condition Test at {target_date}:\nFirms meeting condition: {n_firms:,}\n\n"
        result += sample_df.to_string(index=False) if not sample_df.empty else "(No firms meet this condition)"
        return result
    except Exception as e:
        return f"Error testing condition: {str(e)}"

def get_rate_limited_llm():
    """Get Local Ollama LLM using CrewAI's native LiteLLM wrapper."""
    llm = LLM(
        model="ollama/llama3.1",
        base_url="http://localhost:11434", # Default Ollama port
        timeout=720 # Give the local model extra time to think
    )
    return llm

# ============================================
# ORIGINAL INSTRUCTIONS 
# ============================================
QUANT_BACKSTORY = """
You are an expert quantitative strategist who creates systematic, rule-based trading strategies.
    
CRITICAL DATA UNDERSTANDING:
- 'mve' = log(market value of equity), represents log firm size
- 'bm' = book-to-market ratio (value factor). Understand that a high book-to-market value means undervalued, and a low book-to-market value means overvalued.
- 'mom12m' = 12-month momentum
- ALL features are standardized: mean = 0, standard deviation = 1
- Values are z-scores showing standard deviations from mean

PORTFOLIO SIZE CONSTRAINT:
Do not use excessively large z-scores (like > 3 or < -3) combined with AND statements, as this will result in 0 firms. 
Thresholds between 0.5 and 1.5 are usually best for capturing reasonable tails.

Your task is to develop EXPLICIT, SYSTEMATIC RULES for generating trading signals. 
    
1. EXPLORE THE DATA:
- Identify what constitutes "extreme" values for mve, bm, and mom12m
- Look for natural clustering or breakpoints in the data
- Consider correlations between characteristics
    
2. DEVELOP CLEAR RULES based on economic intuition:
- Keep in mind the market conditions at this date - this might influence the rules you choose.
- Value stocks: Should high bm (cheap/undervalued) be BUY or SELL? What about low bm (expensive)?
- Momentum: Should high mom12m (strong performance) be BUY or SELL?
- Size: Should mve matter for the strategy?
- Combinations: What about value + momentum + size together?
    
3. DEFINE SPECIFIC THRESHOLDS:
 Your output must include exact rules. You can use test_complex_condition to test different combinations.
 Your output must include exact rules like:
- "BUY if: bm > 0.71 AND mom12m > 0.85 AND mve > 0.23" or "BUY if: (bm > 0.57 AND mom12m < 0.82) OR mve > -0.88" 
- "SELL if: bm < -1.25 OR mom12m < -0.98" or "SELL if: (bm < -0.84 OR mve < 0) AND mom12m < -0.97" 
- "HOLD: all other cases"
- However, the above is only AN EXAMPLE - so do not simply copy the format above. You are free to include/exclude as many conditions in the if statements. You are also free to make the conditions as complicated or as simple as you like.
- Be PRECISE in your thresholds, do not choose numbers that are nice or round - you are a quantative strategy developer.
    
4. PROVIDE RATIONALE:
- Why these thresholds?
- What's the economic intuition?
- What patterns did you observed in the data?
    
CRITICAL REQUIREMENTS:
- Rules must be DETERMINISTIC (same inputs → same output)
- Use ONLY z-score comparisons (>, <, AND, OR), but these may be impacted by market conditions.
- Define thresholds for BUY, SELL, and HOLD
- Rules should be implementable as: if (condition) then signal
- Keep in mind that we will use the buy signals to construct a portfolio, so it is better to give too many signals, rather than too few signals.
"""

QUANT_EXPECTED_OUTPUT = """
Complete strategy document with:
1. Data exploration summary
2. Explicit BUY rule with z-score thresholds
3. Explicit SELL rule with z-score thresholds  
4. HOLD rule (default case)
5. Economic rationale for each rule

OUTPUT FORMAT:
===========================================
SYSTEMATIC TRADING RULES
===========================================

Data Exploration Summary:
- [Key statistics and patterns observed]

BUY RULE:
if [ANY complex z-score condition using AND/OR/NOT]:
    signal = BUY

Examples of valid BUY rules:
- "bm > 1.15 AND mom12m > 0.73 AND mve < 1.11" (simple AND)
- "bm > 1.56 OR mom12m > 1.28 OR mve > 1.52" (simple OR)
- "(bm > 1.02 AND mom12m > 0.53) OR mve > 1.59" (combination)
- "bm > 0.83 AND (mom12m > 0.77 OR mve > 1.08)" (nested conditions)

SELL RULE:
if [ANY complex z-score condition]:
    signal = SELL

HOLD RULE:
else:
    signal = HOLD

Rationale:
- [Economic reasoning for BUY rule]
- [Economic reasoning for SELL rule]
- [Expected signal distribution]

===========================================

Be precise, systematic, and data-driven. Your rules will be applied to ~500 firms.
"""

# ============================================
# ACE Agents (max_iter reduced to 5)
# ============================================
generator_agent = Agent(
    role="Quantitative Strategy Developer (Generator)",
    goal="Develop systematic trading rules using the current rulebook, test them against historical forward returns ('ret_fwd_1'), and record reasoning.",
    backstory=QUANT_BACKSTORY + """
    During this training phase, apply the active ACE rulebook to generate BUY/HOLD/SELL signals on historical data. 
    You MUST query 'ret_fwd_1' to evaluate if your signals accurately predicted the forward returns. 
    Record your step-by-step reasoning and explicitly flag which rules from the playbook proved useful and which were misleading when compared against known outcomes.
    """,
    tools=[get_database_schema, query_firm_database, test_complex_condition],
    llm=get_rate_limited_llm(),
    verbose=True,
    max_iter=5, # Reduced from 15
    max_rpm=10
)

reflector_agent = Agent(
    role="Strategy Reflector",
    goal="Analyze the generator's record, identify error patterns, and propose targeted delta updates.",
    backstory="You are an expert performance analyst. Review the Generator's reasoning and the actual realized outcomes ('ret_fwd_1') to diagnose prediction failures. Propose small, targeted 'delta' additions or modifications to the rulebook based on identified patterns.",
    llm=get_rate_limited_llm(),
    verbose=True,
    max_iter=5, # Reduced from 15
    max_rpm=10
)

curator_agent = Agent(
    role="Rulebook Curator",
    goal="Integrate delta updates into the master rulebook while preventing context collapse.",
    backstory="You manage the master rulebook. Integrate proposed deltas from the Reflector by appending new rules, updating existing ones based on performance, and removing duplicates. Modify only small portions of the context to prevent context collapse.",
    llm=get_rate_limited_llm(),
    verbose=True,
    max_iter=5, # Reduced from 15
    max_rpm=10
)

evaluation_agent = Agent(
    role="Quantitative Strategy Developer (Zero-Shot Evaluator)",
    goal="Execute the optimized rulebook on out-of-sample data to generate final trading signals for all S&P 500 firms.",
    backstory=QUANT_BACKSTORY + """
    You must use causal masking to prevent any look-ahead bias. You are in the final evaluation phase. You must take the finalized ACE strategy rulebook and apply it strictly to the current evaluation date. You DO NOT have access to future returns. 
    Output the final explicit thresholds so they can be implemented in pandas.
    """,
    tools=[get_database_schema, query_firm_database, test_complex_condition],
    llm=get_rate_limited_llm(),
    verbose=True,
    max_iter=5, # Reduced from 15
    max_rpm=10
)

# ============================================
# Core Functions
# ============================================
def run_ace_loop(training_start, training_end, iterations=3):
    current_rulebook = """BASE RULEBOOK v1.0: 
    - Focus on finding z-score combinations for value, momentum, and size.
    - Avoid z-scores beyond +/- 2.0 to ensure the portfolio is not empty."""
    
    for i in range(iterations):
        print(f"\n--- ACE Iteration {i+1} ---")
        
        generate_task = Task(
            description=f"""Apply this rulebook to the training data ({training_start} to {training_end}).
            Test conditions using test_complex_condition, query 'ret_fwd_1' to check your hypotheses, and record reasoning.
            CURRENT RULEBOOK:
            {current_rulebook}""",
            expected_output="A detailed log of predictions, the actual 'ret_fwd_1' outcomes, your economic rationale, and flagged rules (useful vs misleading).",
            agent=generator_agent
        )
        
        reflect_task = Task(
            description="""Analyze the Generator's execution log. 
            CRITICAL: If the Generator proposed a rule that returned 0 firms, aggressively flag it as a FAILURE due to extreme thresholds. 
            Output targeted 'delta' updates to lower the thresholds so more firms are captured, while maintaining the economic edge.""",
            expected_output="A structured list of specific delta updates.",
            agent=reflector_agent
        )
        
        curate_task = Task(
            description=f"Integrate the Reflector's delta updates into the current rulebook.\nCURRENT RULEBOOK:\n{current_rulebook}\nOutput the complete updated master rulebook.",
            expected_output="The updated, structured master rulebook with explicit thresholds and economic rationale.",
            agent=curator_agent
        )
        
        ace_crew = Crew(
            agents=[generator_agent, reflector_agent, curator_agent],
            tasks=[generate_task, reflect_task, curate_task],
            process=Process.sequential,
            verbose=False
        )
        current_rulebook = ace_crew.kickoff()
        
    return current_rulebook

def generate_zero_shot_signals(rulebook, eval_date):
    eval_task = Task(
        description=f"""Apply this optimized rulebook strictly to the datadate = '{eval_date}'.
        1. Explore extreme values for each characteristic at this date.
        2. Use test_complex_condition to verify your rule combinations.
        3. Develop systematic rules with specific z-score thresholds.
        
        OPTIMIZED RULEBOOK:
        {rulebook}
        """,
        expected_output=QUANT_EXPECTED_OUTPUT,
        agent=evaluation_agent
    )
    
    eval_crew = Crew(
        agents=[evaluation_agent],
        tasks=[eval_task],
        process=Process.sequential,
        verbose=True
    )
    return eval_crew.kickoff()


# ============================================
# Main Execution
# ============================================
if __name__ == "__main__":
    
    # 1. Setup Database
    # variables = ['datadate', 'ticker', 'bm', 'mom12m', 'mve', 'ret_excess']
    # setup_firm_database('../green cleaned.csv', selected_columns=variables)
    
    # Define your monthly runs. 
    # Adjust the train_start and train_end to reflect your desired rolling window (e.g., trailing 12 months)
    evaluation_runs = [
        #{"eval_date": "2024-01-31", "train_start": "2023-01-01", "train_end": "2023-12-31"},
         #{"eval_date": "2024-02-29", "train_start": "2023-02-01", "train_end": "2024-01-31"},
         {"eval_date": "2024-03-31", "train_start": "2023-03-01", "train_end": "2024-02-29"},
    ]
    
    for run in evaluation_runs:
        eval_date_str = run["eval_date"]
        print(f"\n{'='*70}\n EXECUTING PIPELINE FOR {eval_date_str}\n{'='*70}")
        
        PIPELINE_STATE["current_eval_date"] = eval_date_str
        training_start = run["train_start"]
        training_end = run["train_end"]
        
        # PHASE 1: TRAINING (ACE Loop)
        PIPELINE_STATE["allow_returns"] = True
        print(f"Training ACE Rulebook from {training_start} to {training_end}...")
        optimized_rulebook = run_ace_loop(training_start, training_end, iterations=3)
        
        # PHASE 2: ZERO-SHOT EVAL
        PIPELINE_STATE["allow_returns"] = False
        print(f"\nApplying finalized rulebook zero-shot to {PIPELINE_STATE['current_eval_date']}...")
        final_strategy = generate_zero_shot_signals(optimized_rulebook, PIPELINE_STATE['current_eval_date'])
        
        # Calculate the next month and year for the filename
        eval_dt = datetime.strptime(eval_date_str, "%Y-%m-%d")
        if eval_dt.month == 12:
            next_month = 1
            next_year = eval_dt.year + 1
        else:
            next_month = eval_dt.month + 1
            next_year = eval_dt.year
            
        # Format filename as systematic_strategy_MM_YYYY_llama.txt
        filename = f'systematic_strategy_{next_month:02d}_{next_year}_llama.txt'
        
        with open(filename, 'w') as f:
            f.write(f"--- OPTIMIZED ACE RULEBOOK ---\n{optimized_rulebook}\n\n")
            f.write(f"--- FINAL ZERO-SHOT STRATEGY ---\n{final_strategy}")
            
        print(f"✓ Strategy saved to {filename}")


 EXECUTING PIPELINE FOR 2024-03-31
Training ACE Rulebook from 2023-03-01 to 2024-02-29...

--- ACE Iteration 1 ---


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Quantitative Strategy Developer (Generator)                                                             │
│                                                                                                                 │
│  Task: Apply this rulebook to the training data (2023-03-01 to 2024-02-29).                                     │
│              Test conditions using test_complex_condition, query 'ret_fwd_1' to check your hypotheses, and      │
│  record reasoning.                                                                                              │
│              CURRENT RULEBOOK:                                                                                  │
│              BASE RULEBOOK v1.0:                                                                                │
│      - Focus on finding z-score combinations for value, momentum, and size.                                     │
│      - Avoid z-scores beyond +/- 2.0 to ensure the portfolio is not empty.                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Quantitative Strategy Developer (Generator)                                                             │
│                                                                                                                 │
│  Thought: Thought: I need to understand the data distribution of 'mve', 'bm', and 'mom12m' to identify natural  │
│  clustering or breakpoints.                                                                                     │
│                                                                                                                 │
│  Using Tool: query_firm_database                                                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  {                                                                                                              │
│    "sql_query": "SELECT mve, bm, mom12m FROM firms"                                                             │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Query returned 269255 rows:                                                                                    │
│                                                                                                                 │
│   mve       bm    mom12m                                                                                        │
│   0.0 0.017724 -0.643116                                                                                        │
│   0.0 0.019257 -0.516977                                                                                        │
│   0.0 0.016578 -0.537408                                                                                        │
│   0.0 0.011745 -0.588077                                                                                        │
│   0.0 0.013093 -0.650881                                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Quantitative Strategy Developer (Zero-Shot Evaluator)                                                   │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ===========================================                                                                    │
│  SYSTEMATIC TRADING RULES                                                                                       │
│  ===========================================                                                                    │
│                                                                                                                 │
│  Data Exploration Summary:                                                                                      │
│  - Extreme values for mve: > 1.5 or < -1.5                                                                      │
│  - Extreme values for bm: > 1.2 or < -1.2                                                                       │
│  - Extreme values for mom12m: > 1.1 or < -1.1                                                                   │
│                                                                                                                 │
│  BUY RULE:                                                                                                      │
│  if (bm > 1.2 AND mom12m > 1.1 AND mve < 1.0) OR (bm < -1.2 AND mom12m < -1.1 AND mve > -1.0):                  │
│      signal = BUY                                                                                               │
│                                                                                                                 │
│  SELL RULE:                                                                                                     │
│  if (bm < -1.2 OR mom12m < -1.1 OR mve > 1.5) AND (bm > 1.2 OR mom12m > 1.1 OR mve < -1.5):                     │
│      signal = SELL                                                                                              │
│                                                                                                                 │
│  HOLD RULE:                                                                                                     │
│  else:                                                                                                          │
│      signal = HOLD                                                                                              │
│                                                                                                                 │
│  Rationale:                                                                                                     │
│  - Buy rule: Value stocks with high momentum and low size (cheap/undervalued and strong performance)            │
│  - Sell rule: Value stocks with low momentum and high size (expensive and weak performance) or undervalued      │
│  stocks with high momentum and large size (value trap)                                                          │
│  - Expected signal distribution: Buy signals for value stocks with high momentum and low size, sell signals     │
│  for value stocks with low momentum and high size or undervalued stocks with high momentum and large size.      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 2b68dc68-1e17-4f69-9b30-85cbb8f05977                                                                     │
│  Agent: Quantitative Strategy Developer (Zero-Shot Evaluator)                                                   │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 410a0e55-debe-45b1-bbbe-5cf63c320ed0                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: ===========================================                                                      │
│  SYSTEMATIC TRADING RULES                                                                                       │
│  ===========================================                                                                    │
│                                                                                                                 │
│  Data Exploration Summary:                                                                                      │
│  - Extreme values for mve: > 1.5 or < -1.5                                                                      │
│  - Extreme values for bm: > 1.2 or < -1.2                                                                       │
│  - Extreme values for mom12m: > 1.1 or < -1.1                                                                   │
│                                                                                                                 │
│  BUY RULE:                                                                                                      │
│  if (bm > 1.2 AND mom12m > 1.1 AND mve < 1.0) OR (bm < -1.2 AND mom12m < -1.1 AND mve > -1.0):                  │
│      signal = BUY                                                                                               │
│                                                                                                                 │
│  SELL RULE:                                                                                                     │
│  if (bm < -1.2 OR mom12m < -1.1 OR mve > 1.5) AND (bm > 1.2 OR mom12m > 1.1 OR mve < -1.5):                     │
│      signal = SELL                                                                                              │
│                                                                                                                 │
│  HOLD RULE:                                                                                                     │
│  else:                                                                                                          │
│      signal = HOLD                                                                                              │
│                                                                                                                 │
│  Rationale:                                                                                                     │
│  - Buy rule: Value stocks with high momentum and low size (cheap/undervalued and strong performance)            │
│  - Sell rule: Value stocks with low momentum and high size (expensive and weak performance) or undervalued      │
│  stocks with high momentum and large size (value trap)                                                          │
│  - Expected signal distribution: Buy signals for value stocks with high momentum and low size, sell signals     │
│  for value stocks with low momentum and high size or undervalued stocks with high momentum and large size.      │
│                                                                                                                 │
│                                                       

╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): ✓ Strategy saved to systematic_strategy_04_2024_llama.txt
